In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses/')

import Stellar_sim_funcs as SSF

import os 
import glob

dir = '/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses/Tests/Testing_sim_methods/Plots/T_dep/'

In [ ]:
data_files = glob.glob(os.path.join(dir, '*data*.csv'))
print(data_files)

data = pd.read_csv(data_files[0])

In [ ]:
data.head()


In [ ]:
time_gyr = data['time_Gyr']

no_particles = int((len(data.columns) - 1) / 9)  # Subtract 1 for the time column


kinetic_energy = np.zeros((no_particles, len(time_gyr)))
potential_energy = np.zeros((no_particles, len(time_gyr)))
angular_momentum = np.zeros((no_particles, len(time_gyr)))

x = np.zeros((no_particles, len(time_gyr)))
y = np.zeros((no_particles, len(time_gyr)))
z = np.zeros((no_particles, len(time_gyr)))

vx = np.zeros((no_particles, len(time_gyr)))
vy = np.zeros((no_particles, len(time_gyr)))
vz = np.zeros((no_particles, len(time_gyr)))

for particle in range(no_particles):

    kinetic_energy[particle, :] = data[f'particle_{particle}_kinetic_energy_J']
    potential_energy[particle, :] = data[f'particle_{particle}_potential_energy_J']
    angular_momentum[particle, :] = data[f'particle_{particle}_ang_mom_kg_m2_s']

    x[particle, :] = data[f'particle_{particle}_x_kpc']
    y[particle, :] = data[f'particle_{particle}_y_kpc']
    z[particle, :] = data[f'particle_{particle}_z_kpc']

    vx[particle, :] = data[f'particle_{particle}_v_x_kms']
    vy[particle, :] = data[f'particle_{particle}_v_y_kms']
    vz[particle, :] = data[f'particle_{particle}_v_z_kms']




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib 
matplotlib.rcParams['animation.embed_limit'] = 200  
from IPython.display import HTML

positions_all = np.stack((x, y, z), axis=-1)  # shape (N, T, 3)

N, T, _ = positions_all.shape
dt_Gyr = time_gyr[1] - time_gyr[0]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

# Fix axis limits so the view doesn't jump
xyz_min = positions_all.reshape(-1, 3).min(axis=0) / 3
xyz_max = positions_all.reshape(-1, 3).max(axis=0) / 3
ax.set_xlim(xyz_min[0], xyz_max[0])
ax.set_ylim(xyz_min[1], xyz_max[1])
ax.set_zlim(xyz_min[2], xyz_max[2])
ax.set_xlabel('X [kpc]'); ax.set_ylabel('Y [kpc]'); ax.set_zlabel('Z [kpc]')

# One trail line + one head point per particle
trails = [ax.plot([], [], [], lw=0.8)[0] for _ in range(N)]
heads  = [ax.plot([], [], [], 'o', ms=3)[0] for _ in range(N)]

# Time readout
time_text = ax.text2D(0.02, 0.98, '', transform=ax.transAxes,
                      verticalalignment='top', fontsize=11,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

trail_len = 50  # frames of history to draw; set to T for full trail

def update(frame):
    start = max(0, frame - trail_len)
    for i in range(N):
        seg = positions_all[i, start:frame+1] 
        trails[i].set_data(seg[:, 0], seg[:, 1])
        trails[i].set_3d_properties(seg[:, 2])
        p = positions_all[i, frame]
        heads[i].set_data([p[0]], [p[1]])
        heads[i].set_3d_properties([p[2]])
    time_text.set_text(f't = {frame * dt_Gyr:.3f} Gyr')
    return trails + heads + [time_text]

anim = FuncAnimation(fig, update, frames=T, interval=40, blit=False)
HTML(anim.to_jshtml())   
#anim.save('orbits.mp4', fps=25)


In [ ]:
r_all = np.linalg.norm(positions_all, axis=-1)  # shape (N, T)

r_average = r_all.mean(axis=0)

plt.figure(figsize=(8, 5))
for i in range(N-1):
    plt.plot(time_gyr, r_all[i], color='red', alpha=0.5)
plt.plot(time_gyr, r_all[-1], label = 'individual particles', color='red', alpha=0.5)
plt.plot(time_gyr, r_average, label='Average', color='black', linewidth=2)
plt.xlabel('Time [Gyr]')
plt.axvline(x=1, color='green', linestyle='--', label='End of ramp phase', alpha = 0.5)
plt.ylabel('Distance from Galactic Center [kpc]')
plt.title('Radial Distance vs Time')
plt.legend()
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

vr, vtheta, vphi = SSF.Cartesian_to_sph_vel_np(x, y, z, vx, vy, vz)


velocity_disp_x = np.std(vx, axis=0)
velocity_disp_y = np.std(vy, axis=0)
velocity_disp_z = np.std(vz, axis=0)

velocity_disp_tot = np.sqrt(velocity_disp_x**2 + velocity_disp_y**2 + velocity_disp_z**2)


plt.plot(time_gyr, velocity_disp_tot, label='Total Velocity Dispersion', color='black', linewidth=2)
plt.xlabel('Time [Gyr]')
plt.axvline(x=1, color='red', linestyle='--', label='End of ramp phase', alpha = 0.5)
plt.ylabel('Velocity Dispersion [km/s]')
plt.title('Velocity Dispersion vs Time')
plt.legend()
plt.grid()
plt.show()


variance_v = np.var(np.sqrt(vx**2 + vy**2 + vz**2), axis=0)

plt.figure(figsize=(10, 6))
plt.plot(time_gyr, variance_v, label='Velocity Variance', color='orange', linewidth=2)
plt.xlabel('Time [Gyr]')
plt.axvline(x=1, color='red', linestyle='--', label='End of ramp phase', alpha = 0.5)
plt.ylabel('Velocity [km/s]')
plt.title('Mean Velocity and Variance vs Time')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(N):
    plt.plot(time_gyr, kinetic_energy[i] + potential_energy[i], label=f'Particle {i} Total Energy', alpha=0.5)

plt.xlabel('Time [Gyr]')
plt.ylabel('Energy [J]')
plt.title('Energy vs Time')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 6))

for i in range(N):
    plt.plot(time_gyr, angular_momentum[i], label=f'Particle {i} Angular Momentum', alpha=0.5)
plt.xlabel('Time [Gyr]')
plt.ylabel('Angular Momentum [kg m^2/s]')
plt.title('Angular Momentum vs Time')
plt.legend()
plt.grid()
plt.show()